# Проверка данных и обучения Orthrus
Выбери конфиг ниже. Нужны подготовленные данные и доступ к базовой модели.
Ноутбук считает loss и градиенты без шага оптимизатора; на HF ничего не отправляет.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config
CONFIG = os.environ.get("ORTHRUS_CONFIG", "configs/qwen3-0.6b.yaml")
cfg = load_config(CONFIG)
print("Experiment:", cfg.run_name)


In [ ]:
import numpy as np
from src.dataset.data import fetch_raw, load_manifest, load_packed
from src.hub import Bucket
from src.train.train import check_teacher

bucket = Bucket(cfg.hf.bucket, upload=False)
manifest = load_manifest(cfg, bucket)
check_teacher(cfg, manifest)
raw = fetch_raw(cfg, bucket)
ids, masks, fingerprint = load_packed(raw, raw.parent / "packed", cfg.data.seq_len, cfg.data.seed)
assert ids.shape == masks.shape
assert ids.ndim == 2 and ids.shape[1] == cfg.data.seq_len
assert set(np.unique(masks)) <= {0, 1}
print("Packed rows:", ids.shape, "Fingerprint:", fingerprint)
print("Assistant-token fraction:", float(masks.mean()))


## Модель и один backward
Обучаемы только diffusion attention и q/k-нормировки. На CPU проверка идёт в float32.

In [ ]:
import torch
from src.train.checkpoint import load_model
from src.train.loss import sample_anchors, orthrus_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
model, tokenizer, trainable = load_model(cfg.model, device, dtype)
names = [n for n, p in model.named_parameters() if p.requires_grad]
assert names and all("_diff." in n for n in names)
print("Trainable parameters:", sum(p.numel() for p in trainable))
print("Example:", tokenizer.decode(ids[0].tolist(), skip_special_tokens=False)[:1000])

batch_ids = torch.tensor(np.array(ids[:1]), dtype=torch.long, device=device)
batch_mask = torch.tensor(np.array(masks[:1]), dtype=torch.bool, device=device)
anchors, valid = sample_anchors(batch_mask, cfg.model.block_size,
                                min(4, cfg.train.num_anchor_blocks),
                                torch.Generator(device=device).manual_seed(0))
model.train()
with torch.autocast(device.type, torch.bfloat16, enabled=device.type == "cuda"):
    loss, stats = orthrus_loss(model, batch_ids, batch_mask, anchors, valid,
                               min(64, cfg.train.kl_chunk_size), gamma=cfg.train.loss_gamma)
loss.backward()
assert torch.isfinite(loss) and int(stats["tokens"]) > 0
assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
print("Loss:", float(loss.detach()), "Supervised tokens:", int(stats["tokens"]))
model.zero_grad(set_to_none=True)


## Совпадение greedy-генерации с AR
Для строгой проверки используй float32; при bf16 возможны расхождения из-за округления близких логитов.

In [ ]:
from src.test.benchmark import ar_generate, eos_token_ids

model.eval()
prompt = torch.tensor([ids[0, :min(16, cfg.data.seq_len)].tolist()], device=device)
eos = eos_token_ids(model, tokenizer)
with torch.inference_mode():
    output, stats = model.diffusion_generate(prompt, 16, eos_token_id=list(eos))
    reference = ar_generate(model, prompt, 16, eos)
print("Matches AR:", output[0, prompt.shape[1]:].tolist() == reference)
print("TPF:", stats.new_tokens / stats.forward_passes)
